### RNN Model 

In [1]:
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense,Dropout,BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### load the datasets

In [3]:
X_train_padded = np.load("models/X_train_padded.npy")
X_val_padded = np.load("models/X_val_padded.npy")
X_test_padded = np.load("models/X_test_padded.npy")
y_train = np.load("models/y_train.npy")
y_val = np.load("models/y_val.npy")
y_test = np.load("models/y_test.npy")

sequence_length = X_train_padded.shape[1]
vocab_size = 20000

print(X_train_padded.shape)
print(X_val_padded.shape)
print(X_test_padded.shape)

(34705, 200)
(7439, 200)
(7438, 200)


In [54]:
lstm_results = []

def evaluate_lstm(model, experiment_name, X_test_data=None):

    # Use 200-token test data by default
    if X_test_data is None:
        X_test_data = X_test_padded

    # Prediction probabilities
    y_prob = model.predict(
        X_test_data,
        verbose=0
    ).ravel()

    # Convert probabilities to classes
    y_pred = (y_prob >= 0.5).astype(int)

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store results
    lstm_results.append({
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    # Display results
    print(f"\n{experiment_name}")
    print("-" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### Baseline LSTM

In [5]:
lstm_model = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,613,633 (9.97 MB)

 Trainable params: 2,613,633 (9.97 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history_lstm = lstm_model.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 63s 57ms/step - accuracy: 0.5122 - loss: 0.6918 - val_accuracy: 0.5161 - val_loss: 0.6892
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 61s 56ms/step - accuracy: 0.5729 - loss: 0.6438 - val_accuracy: 0.8523 - val_loss: 0.3780
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 61s 56ms/step - accuracy: 0.8899 - loss: 0.2737 - val_accuracy: 0.8892 - val_loss: 0.2753
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 61s 56ms/step - accuracy: 0.9502 - loss: 0.1476 - val_accuracy: 0.8820 - val_loss: 0.3244
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 63s 58ms/step - accuracy: 0.9771 - loss: 0.0788 - val_accuracy: 0.8806 - val_loss: 0.3818


In [55]:
evaluate_lstm(
    lstm_model,
    "LSTM - Baseline"
)
lstm_model.save("models/lstm_baseline.keras")


LSTM - Baseline
----------------------------------------
Accuracy : 0.8768
Precision: 0.8767
Recall   : 0.8781
F1 Score : 0.8774
ROC-AUC  : 0.9413


### Embedding Dimension

In [11]:
lstm_embedding = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=256
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_embedding.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [12]:
history_embedding = lstm_embedding.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 105s 96ms/step - accuracy: 0.5344 - loss: 0.6864 - val_accuracy: 0.5279 - val_loss: 0.6875
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 106s 97ms/step - accuracy: 0.6688 - loss: 0.5556 - val_accuracy: 0.8587 - val_loss: 0.3740
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 104s 96ms/step - accuracy: 0.8976 - loss: 0.2666 - val_accuracy: 0.8820 - val_loss: 0.3056
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 104s 96ms/step - accuracy: 0.9442 - loss: 0.1585 - val_accuracy: 0.8822 - val_loss: 0.3297
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 105s 97ms/step - accuracy: 0.9708 - loss: 0.0906 - val_accuracy: 0.8732 - val_loss: 0.3975


In [56]:
evaluate_lstm(
    lstm_embedding,
    "LSTM - Embedding 256"
)

lstm_embedding.save("models/lstm_embedding_256.keras")


LSTM - Embedding 256
----------------------------------------
Accuracy : 0.8668
Precision: 0.8512
Recall   : 0.8902
F1 Score : 0.8702
ROC-AUC  : 0.9371


### Sequential Length

In [15]:
with open("models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

vocab_size = len(tokenizer.word_index) + 1

In [16]:
X_train_text = pd.read_pickle("models/X_train_text.pkl")
X_val_text = pd.read_pickle("models/X_val_text.pkl")
X_test_text = pd.read_pickle("models/X_test_text.pkl")

print(X_train_text.shape)
print(X_val_text.shape)
print(X_test_text.shape)

(34705,)
(7439,)
(7438,)


In [17]:
X_train_sequences = tokenizer.texts_to_sequences(X_train_text)
X_val_sequences = tokenizer.texts_to_sequences(X_val_text)
X_test_sequences = tokenizer.texts_to_sequences(X_test_text)

In [18]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_padded_300 = pad_sequences(
    X_train_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_val_padded_300 = pad_sequences(
    X_val_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_test_padded_300 = pad_sequences(
    X_test_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

In [19]:
lstm_sequence = Sequential([
    Input(shape=(300,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_sequence.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [20]:
history_sequence = lstm_sequence.fit(
    X_train_padded_300,
    y_train,
    validation_data=(X_val_padded_300, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 112s 103ms/step - accuracy: 0.5052 - loss: 0.6942 - val_accuracy: 0.4985 - val_loss: 0.6937
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.5110 - loss: 0.6917 - val_accuracy: 0.5077 - val_loss: 0.6921
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 110s 102ms/step - accuracy: 0.5207 - loss: 0.6732 - val_accuracy: 0.5107 - val_loss: 0.6974
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.6707 - loss: 0.5383 - val_accuracy: 0.8578 - val_loss: 0.3447
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 112s 103ms/step - accuracy: 0.9112 - loss: 0.2313 - val_accuracy: 0.8942 - val_loss: 0.2728


In [57]:
evaluate_lstm(
    lstm_sequence,
    "LSTM - Sequence Length 300",
    X_test_padded_300
)
lstm_sequence.save("models/lstm_sequence_300.keras")


LSTM - Sequence Length 300
----------------------------------------
Accuracy : 0.8911
Precision: 0.8835
Recall   : 0.9020
F1 Score : 0.8926
ROC-AUC  : 0.9530


### Hidden Units

In [22]:
lstm_hidden = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        128,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_hidden.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [23]:
history_hidden = lstm_hidden.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 146s 134ms/step - accuracy: 0.5075 - loss: 0.6921 - val_accuracy: 0.5249 - val_loss: 0.6858
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 145s 134ms/step - accuracy: 0.7158 - loss: 0.5143 - val_accuracy: 0.8537 - val_loss: 0.3472
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 144s 132ms/step - accuracy: 0.9068 - loss: 0.2464 - val_accuracy: 0.8654 - val_loss: 0.3198
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 143s 132ms/step - accuracy: 0.9545 - loss: 0.1389 - val_accuracy: 0.8771 - val_loss: 0.3392
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 143s 132ms/step - accuracy: 0.9767 - loss: 0.0745 - val_accuracy: 0.8722 - val_loss: 0.4122


In [58]:
evaluate_lstm(
    lstm_hidden,
    "RNN - Hidden Units 128"
)

lstm_hidden.save("models/lstm_hidden_128.keras")


RNN - Hidden Units 128
----------------------------------------
Accuracy : 0.8763
Precision: 0.8622
Recall   : 0.8969
F1 Score : 0.8792
ROC-AUC  : 0.9368


### Dropout

In [25]:
lstm_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [26]:
history_dropout = lstm_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 87s 79ms/step - accuracy: 0.5118 - loss: 0.6924 - val_accuracy: 0.5153 - val_loss: 0.6878
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 84s 78ms/step - accuracy: 0.5437 - loss: 0.6622 - val_accuracy: 0.5370 - val_loss: 0.6801
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.7797 - loss: 0.4383 - val_accuracy: 0.8746 - val_loss: 0.3317
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.9231 - loss: 0.2176 - val_accuracy: 0.8777 - val_loss: 0.3256
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9633 - loss: 0.1150 - val_accuracy: 0.8653 - val_loss: 0.3886


In [59]:
evaluate_lstm(
    lstm_dropout,
    "lstm - Dropout"
)

lstm_dropout.save("models/lstm_dropout.keras")


lstm - Dropout
----------------------------------------
Accuracy : 0.8617
Precision: 0.8960
Recall   : 0.8194
F1 Score : 0.8560
ROC-AUC  : 0.9353


### Different Optimizer

In [29]:
lstm_rmsprop = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [30]:

history_rmsprop = lstm_rmsprop.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 74s 67ms/step - accuracy: 0.5073 - loss: 0.6936 - val_accuracy: 0.5200 - val_loss: 0.6938
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 67ms/step - accuracy: 0.5382 - loss: 0.6730 - val_accuracy: 0.5576 - val_loss: 0.6550
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 67ms/step - accuracy: 0.5805 - loss: 0.6416 - val_accuracy: 0.7031 - val_loss: 0.5961
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 74s 68ms/step - accuracy: 0.8390 - loss: 0.4077 - val_accuracy: 0.8453 - val_loss: 0.3503
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 67ms/step - accuracy: 0.8964 - loss: 0.2641 - val_accuracy: 0.8766 - val_loss: 0.3012


In [60]:

evaluate_lstm(
    lstm_rmsprop,
    "LSTM - RMSprop"
)

lstm_rmsprop.save("models/lstm_rmsprop.keras")


LSTM - RMSprop
----------------------------------------
Accuracy : 0.8727
Precision: 0.8320
Recall   : 0.9352
F1 Score : 0.8806
ROC-AUC  : 0.9502


### Batch Normalization

In [32]:
lstm_batchnorm = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    BatchNormalization(),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_batchnorm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [33]:

history_batchnorm = lstm_batchnorm.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.5225 - loss: 0.6867 - val_accuracy: 0.5128 - val_loss: 0.6783
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.7833 - loss: 0.4295 - val_accuracy: 0.6939 - val_loss: 0.9031
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9230 - loss: 0.2020 - val_accuracy: 0.8864 - val_loss: 0.2840
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.9616 - loss: 0.1111 - val_accuracy: 0.8472 - val_loss: 0.5066
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.9797 - loss: 0.0619 - val_accuracy: 0.8540 - val_loss: 0.5195


In [61]:

evaluate_lstm(
    lstm_batchnorm,
    "LSTM - Batch Normalization"
)

lstm_batchnorm.save("models/lstm_batchnorm.keras")


LSTM - Batch Normalization
----------------------------------------
Accuracy : 0.8555
Precision: 0.9158
Recall   : 0.7841
F1 Score : 0.8449
ROC-AUC  : 0.9399


### Learning Rate

In [35]:
lstm_learning_rate = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_learning_rate.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [36]:

history_learning_rate = lstm_learning_rate.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)



Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 84s 76ms/step - accuracy: 0.5094 - loss: 0.6908 - val_accuracy: 0.5608 - val_loss: 0.6723
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.5485 - loss: 0.6778 - val_accuracy: 0.5439 - val_loss: 0.6709
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.5623 - loss: 0.6587 - val_accuracy: 0.5624 - val_loss: 0.6618
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.6804 - loss: 0.5853 - val_accuracy: 0.7459 - val_loss: 0.5410
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.7384 - loss: 0.5503 - val_accuracy: 0.7856 - val_loss: 0.5018


In [62]:
evaluate_lstm(
    lstm_learning_rate,
    "LSTM - Learning Rate 0.0001"
)

lstm_learning_rate.save("models/LSTM_learning_rate.keras")


LSTM - Learning Rate 0.0001
----------------------------------------
Accuracy : 0.7856
Precision: 0.8634
Recall   : 0.6804
F1 Score : 0.7610
ROC-AUC  : 0.8122


### Batch Size

In [38]:
lstm_batch_size = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_batch_size.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [39]:

history_batch_size = lstm_batch_size.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)


Epoch 1/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 66s 120ms/step - accuracy: 0.5197 - loss: 0.6855 - val_accuracy: 0.5255 - val_loss: 0.6746
Epoch 2/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 65s 120ms/step - accuracy: 0.5529 - loss: 0.6442 - val_accuracy: 0.5464 - val_loss: 0.6667
Epoch 3/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 65s 120ms/step - accuracy: 0.8594 - loss: 0.3231 - val_accuracy: 0.8715 - val_loss: 0.3214
Epoch 4/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 66s 121ms/step - accuracy: 0.9424 - loss: 0.1621 - val_accuracy: 0.8832 - val_loss: 0.3387
Epoch 5/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 65s 120ms/step - accuracy: 0.9718 - loss: 0.0864 - val_accuracy: 0.8798 - val_loss: 0.3968


In [63]:

evaluate_lstm(
    lstm_batch_size,
    "LSTM - Batch Size 64"
)

lstm_batch_size.save("models/LSTM_batch_size_64.keras")


LSTM - Batch Size 64
----------------------------------------
Accuracy : 0.8810
Precision: 0.8832
Recall   : 0.8792
F1 Score : 0.8812
ROC-AUC  : 0.9454


### Early Stopping

In [41]:
lstm_early_stopping = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_early_stopping.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


In [42]:

history_early_stopping = lstm_early_stopping.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)



Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.5162 - loss: 0.6884 - val_accuracy: 0.5088 - val_loss: 0.6885
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.6794 - loss: 0.5341 - val_accuracy: 0.8668 - val_loss: 0.3186
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9064 - loss: 0.2442 - val_accuracy: 0.8887 - val_loss: 0.2819
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9540 - loss: 0.1332 - val_accuracy: 0.8781 - val_loss: 0.3566
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.9786 - loss: 0.0678 - val_accuracy: 0.8705 - val_loss: 0.4696


In [64]:
evaluate_lstm(
    lstm_early_stopping,
    "LSTM - Early Stopping"
)

lstm_early_stopping.save(
    "models/LSTM_early_stopping.keras"
)


LSTM - Early Stopping
----------------------------------------
Accuracy : 0.8849
Precision: 0.8815
Recall   : 0.8904
F1 Score : 0.8859
ROC-AUC  : 0.9517


### Learning Rate Scheduling

In [44]:
lstm_scheduler = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_scheduler.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)


In [45]:

history_scheduler = lstm_scheduler.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    callbacks=[reduce_lr],
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.5184 - loss: 0.6864 - val_accuracy: 0.5346 - val_loss: 0.6727 - learning_rate: 0.0010
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.6120 - loss: 0.6252 - val_accuracy: 0.8064 - val_loss: 0.4739 - learning_rate: 0.0010
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.8763 - loss: 0.3088 - val_accuracy: 0.8769 - val_loss: 0.3033 - learning_rate: 0.0010
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9406 - loss: 0.1672 - val_accuracy: 0.8802 - val_loss: 0.3228 - learning_rate: 0.0010
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 84s 77ms/step - accuracy: 0.9775 - loss: 0.0775 - val_accuracy: 0.8771 - val_loss: 0.3767 - learning_rate: 5.0000e-04


In [65]:

evaluate_lstm(
    lstm_scheduler,
    "LSTM - Learning Rate Scheduling"
)

lstm_scheduler.save(
    "models/lstm_learning_rate_scheduler.keras"
)


LSTM - Learning Rate Scheduling
----------------------------------------
Accuracy : 0.8742
Precision: 0.8762
Recall   : 0.8725
F1 Score : 0.8744
ROC-AUC  : 0.9438


### Recurrent Dropout

In [48]:
lstm_recurrent_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        64,
        activation="tanh",
        recurrent_dropout=0.3
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_recurrent_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [50]:
history_recurrent_dropout = lstm_recurrent_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 116s 107ms/step - accuracy: 0.5163 - loss: 0.6908 - val_accuracy: 0.5216 - val_loss: 0.6852
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 117s 108ms/step - accuracy: 0.5689 - loss: 0.6500 - val_accuracy: 0.7977 - val_loss: 0.4635
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 117s 108ms/step - accuracy: 0.8758 - loss: 0.3026 - val_accuracy: 0.8875 - val_loss: 0.2831
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 117s 107ms/step - accuracy: 0.9440 - loss: 0.1584 - val_accuracy: 0.8844 - val_loss: 0.3039
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 117s 107ms/step - accuracy: 0.9740 - loss: 0.0841 - val_accuracy: 0.8761 - val_loss: 0.4019


In [66]:

evaluate_lstm(
    lstm_recurrent_dropout,
    "LSTM - Recurrent Dropout"
)

lstm_recurrent_dropout.save(
    "models/LSTM_recurrent_dropout.keras"
)


LSTM - Recurrent Dropout
----------------------------------------
Accuracy : 0.8760
Precision: 0.8773
Recall   : 0.8754
F1 Score : 0.8764
ROC-AUC  : 0.9391


In [67]:
lstm_results_df = pd.DataFrame(lstm_results)

lstm_results_df

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM - Baseline,0.876849,0.876705,0.878114,0.877409,0.941283
1,LSTM - Embedding 256,0.866765,0.851178,0.890169,0.870237,0.937111
2,LSTM - Sequence Length 300,0.891100,0.883495,0.901956,0.892630,0.952989
3,RNN - Hidden Units 128,0.876311,0.862220,0.896866,0.879202,0.936811
4,lstm - Dropout,0.861656,0.896016,0.819448,0.856024,0.935343
5,LSTM - RMSprop,0.872681,0.831983,0.935173,0.880565,0.950200
6,LSTM - Batch Normalization,0.855472,0.915832,0.784088,0.844855,0.939931
7,LSTM - Learning Rate 0.0001,0.785561,0.863358,0.680418,0.761049,0.812209
8,LSTM - Batch Size 64,0.881016,0.883208,0.879186,0.881192,0.945447
9,LSTM - Early Stopping,0.884915,0.881464,0.890437,0.885928,0.951669


In [68]:
lstm_results_df.to_csv("models/LSTM_comparison_table.keras",index=False)